# 05 — Hotspot Analysis and Maps

Run spatial autocorrelation, rank the most vulnerable municipalities, and export publication-ready maps.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import (
    RAW_DIR, PROCESSED_DIR, OUTPUT_DIR,
    BOUNDARIES_WFS, HEALTH_WFS, SOCIAL_WFS, POPULATION_CSV,
    METRIC_CRS, MAP_CRS, GEOGRAPHIC_CRS,
    BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER,
    MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL, TOTAL_POP_COL,
)
from scripts.data_sources import SOURCES
from scripts.wfs_utils import discover_wfs_layers, load_wfs_layer, download_csv, save_geodataframe, save_dataframe
from scripts.population_utils import normalize_columns, build_population_65_plus
from scripts.analysis_utils import standardize_geodataframes, build_vulnerability_index, spatial_autocorrelation, top_ranked
from scripts.plotting_utils import save_choropleth
from scripts.export_utils import export_geodataframe, export_dataframe

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gdf = gpd.read_file(PROCESSED_DIR / "municipality_vulnerability.gpkg")
gdf = gdf.to_crs(METRIC_CRS)

In [ ]:
gdf, w, moran, g_local = spatial_autocorrelation(gdf, "vulnerability_index")
print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

In [ ]:
rankings = top_ranked(gdf, "vulnerability_index", n=20)
rankings.to_csv(OUTPUT_DIR / "top_20_vulnerable_municipalities.csv", index=False)
rankings

In [ ]:
save_choropleth(
    gdf,
    column="vulnerability_index",
    output_path=OUTPUT_DIR / "vulnerability_index_map.png",
    title="Healthcare Accessibility & Ageing Vulnerability Index — Spain"
)
print("Saved static map to data/outputs/vulnerability_index_map.png")

In [ ]:
out = gdf.to_crs("EPSG:4326")
import folium

m = folium.Map(location=[40.4, -3.7], zoom_start=6, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=out.to_json(),
    data=out,
    columns=[MUNICIPALITY_CODE_COL, "vulnerability_index"],
    key_on=f"feature.properties.{MUNICIPALITY_CODE_COL}",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.2,
    legend_name="Vulnerability Index",
).add_to(m)

m.save(str(OUTPUT_DIR / "interactive_vulnerability_map.html"))
m